# RankSEG with Hugging Face Transformers

This notebook demonstrates a focused Colab integration path for applying **RankSEG** on top of a standard Hugging Face semantic segmentation workflow.

## What RankSEG changes

RankSEG does not replace model inference. It replaces the final decision step that turns dense per-class scores into a discrete segmentation map. In place of plain `argmax`, RankSEG applies a metric-aware post-processing procedure designed to improve the final mask with respect to overlap-based objectives such as **Dice**.

## What this notebook covers

1. Run the original `jonathandinu/face-parsing` example as a baseline.
2. Keep the same preprocessing, model loading, and forward pass.
3. Swap only the last post-processing step for `rankseg.transformers.postprocess(...)`.
4. Compare the baseline output and the RankSEG result side by side.

The emphasis is deliberate: this notebook is not about introducing a new inference stack. It is about showing that, for supported `transformers` segmentation outputs, RankSEG can be inserted as a narrow post-processing layer with minimal disruption to the surrounding code.


In [ ]:
# Colab setup
%pip install -q uv
!uv pip install -U rankseg
!uv pip install -q transformers pillow requests matplotlib


In [ ]:
import gc
import torch

def cleanup_runtime(*names):
    """Release named globals and clear CUDA cache between notebook stages."""
    for name in names:
        globals().pop(name, None)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()


## 1. Official Hugging Face baseline

The next code cell follows the original `jonathandinu/face-parsing` usage pattern closely. The steps are conventional and familiar:

- preprocess the image with `SegformerImageProcessor`
- run the model forward pass
- resize logits back to the original image resolution
- obtain the final label map with `argmax`

Reference model card: [jonathandinu/face-parsing](https://huggingface.co/jonathandinu/face-parsing)


In [ ]:
import torch
from torch import nn
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

from PIL import Image
import matplotlib.pyplot as plt
import requests

device = "cuda" if torch.cuda.is_available() else "cpu"

# load models
image_processor = SegformerImageProcessor.from_pretrained("jonathandinu/face-parsing")
model = SegformerForSemanticSegmentation.from_pretrained("jonathandinu/face-parsing")
model.to(device)

# expects a PIL.Image or torch.Tensor
url = "https://images.unsplash.com/photo-1539571696357-5a69c17a67c6"
image = Image.open(requests.get(url, stream=True).raw)

# run inference on image
inputs = image_processor(images=image, return_tensors="pt").to(device)
outputs = model(**inputs)
logits = outputs.logits  # shape (batch_size, num_labels, ~height/4, ~width/4)

# resize output to match input image dimensions
upsampled_logits = nn.functional.interpolate(logits,
                size=image.size[::-1], # H x W
                mode='bilinear',
                align_corners=False)

# get label masks
labels = upsampled_logits.argmax(dim=1)[0]

# move to CPU to visualize in matplotlib
labels_viz = labels.cpu().numpy()
plt.imshow(labels_viz)
plt.show()


In [ ]:
# keep the baseline visualization, and release the heavier tensors before the RankSEG stage
cleanup_runtime(
    "model",
    "image_processor",
    "image",
    "inputs",
    "outputs",
    "logits",
    "upsampled_logits",
    "labels",
)


## 2. RankSEG drop-in version

This cell intentionally repeats the same inference structure. The processor, model, image preparation, and forward pass all remain unchanged.

The integration point is a single hand-off after `outputs = model(**inputs)`. At that point, `rankseg.transformers.postprocess(...)` takes over the final stage of the pipeline. For supported Hugging Face segmentation outputs, it restores semantic probabilities according to the corresponding `transformers` post-processing behavior and then applies RankSEG at the original image resolution.


In [ ]:
import torch
from torch import nn
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from rankseg.transformers import postprocess

from PIL import Image
import matplotlib.pyplot as plt
import requests

device = "cuda" if torch.cuda.is_available() else "cpu"

# load models
image_processor = SegformerImageProcessor.from_pretrained("jonathandinu/face-parsing")
model = SegformerForSemanticSegmentation.from_pretrained("jonathandinu/face-parsing")
model.to(device)

# expects a PIL.Image or torch.Tensor
url = "https://images.unsplash.com/photo-1539571696357-5a69c17a67c6"
image = Image.open(requests.get(url, stream=True).raw)

# run inference on image
inputs = image_processor(images=image, return_tensors="pt").to(device)
outputs = model(**inputs)

# get label masks
rankseg_labels = postprocess(
    outputs,
    target_sizes=image.size[::-1],
    rankseg_kwargs={"metric": "dice"},
)[0]

# move to CPU to visualize in matplotlib
rankseg_labels_viz = rankseg_labels.cpu().numpy()
plt.imshow(rankseg_labels_viz)
plt.show()


In [ ]:
# keep the RankSEG visualization for comparison, and release the second-stage tensors
cleanup_runtime(
    "model",
    "image_processor",
    "inputs",
    "outputs",
    "rankseg_labels",
)


## 3. Visual comparison

The comparison below is meant to answer a narrow question: **what changes when only the final post-processing step is replaced, while everything upstream remains the same?**

The four panels show:

1. the input image
2. the official Hugging Face `argmax` baseline
3. the RankSEG post-processed result
4. a binary difference map between the two outputs


In [ ]:
difference_viz = (labels_viz != rankseg_labels_viz).astype("uint8")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(image)
axes[0].set_title("Input image")
axes[1].imshow(labels_viz)
axes[1].set_title("Baseline argmax")
axes[2].imshow(rankseg_labels_viz)
axes[2].set_title("RankSEG postprocess")
axes[3].imshow(difference_viz, cmap="gray", vmin=0, vmax=1)
axes[3].set_title("Difference map")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Takeaway

This notebook illustrates the current integration contract between RankSEG and Hugging Face `transformers` in its simplest form:

- keep the standard preprocessing and forward pass
- treat model `outputs` as the integration boundary
- call `rankseg.transformers.postprocess(...)` with the original `target_sizes`
